# 🥉 Bronze Layer — Crypto Raw Ingestion

**Proyecto:** CoinMarketCap Analytics Platform  
**Capa:** Bronze  
**Fuente:** CoinGecko API (https://api.coingecko.com)  
**Responsabilidad:** Ingestar datos crudos de las top 10 criptomonedas 
y persistirlos en Delta Lake SIN transformaciones.  

**Regla Bronze:** los datos se guardan exactamente como llegan de la fuente.

In [0]:
%python
#Librerías estándar de Python

import requests # para llamar la API HTTP
import json # Para manejar el json que nos devuelve
from datetime import datetime # Para registrar cuando ingestamos 

# PySpark — ya viene instalado en Databricks, no necesitás pip install
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, LongType, TimestampType
)
from pyspark.sql.functions import current_timestamp, lit

In [0]:
%python
spark.sql("""
SELECT 
    current_catalog() AS catalogo_actual,
    current_schema() AS schema_actual
""").show()

In [0]:
%python
spark.sql("CREATE SCHEMA IF NOT EXISTS crypto_project")
spark.sql("USE SCHEMA crypto_project")

In [0]:
%python
api_url = "https://api.coingecko.com/api/v3/coins/markets"

params = {
    "vs_currency": "usd",
    "order": "market_cap_desc",
    "per_page": 10,
    "page": 1,
    "sparkline": "false",
    "price_change_percentage": "24h"
}

response = requests.get(api_url, params=params, timeout=30)
response.raise_for_status()

data = response.json()

print(type(data))
print(len(data))
print(json.dumps(data[0], indent=2)[:1000])

In [0]:
%python

batch_ingestion_ts = datetime.utcnow()

raw_records = []

for item in data:
    raw_records.append({
        "coin_id": item.get("id"),
        "symbol": item.get("symbol"),
        "raw_json": json.dumps(item),
        "source_system": "coingecko",
        "api_endpoint": api_url,
        "ingestion_ts": datetime.utcnow()
    })

schema = StructType([
    StructField("coin_id", StringType(), True),
    StructField("symbol", StringType(), True),
    StructField("raw_json", StringType(), True),
    StructField("source_system", StringType(), True),
    StructField("api_endpoint", StringType(), True),
    StructField("ingestion_ts", TimestampType(), True)
])

bronze_df = spark.createDataFrame(raw_records, schema)

display(bronze_df)

In [0]:
%python
bronze_table = "crypto_project.bronze_crypto_markets_raw"

bronze_df.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable(bronze_table)

print(f"Datos guardados correctamente en la tabla: {bronze_table}")

In [0]:
%python
spark.sql("""
SELECT *
FROM crypto_project.bronze_crypto_markets_raw
ORDER BY ingestion_ts DESC
LIMIT 10
""").show(truncate=False)

In [0]:
%python
spark.sql("""
DESCRIBE HISTORY crypto_project.bronze_crypto_markets_raw
""").show(truncate=False)

In [0]:
%python
history_df = spark.sql("""
DESCRIBE HISTORY crypto_project.bronze_crypto_markets_raw
""")

display(history_df)

In [0]:
%python
display(
    spark.sql("""
    SELECT *
    FROM crypto_project.bronze_crypto_markets_raw
    ORDER BY ingestion_ts DESC
    LIMIT 10
    """)
)

In [0]:
%python
spark.sql("""
SELECT COUNT(*) AS total_registros
FROM crypto_project.bronze_crypto_markets_raw
""").show()